# Transform Yield Data

Reads raw maize yield CSVs for each location and reshapes them from wide to long form.
Columns like `A-N0 Zrnje` are split into `management` (A/B/C), `fertilization` (N0–N3), and `product` (Zrnje/Slama).
Outputs are saved to `data/interim/yield/`.

In [1]:
from aquacrop_slovenia import config
import pandas as pd

In [5]:
locations = {
    "jablje": config.RAW_YIELD_DIR / "maize-jablje.csv",
    "rakican": config.RAW_YIELD_DIR / "maize-rakican.csv",
}

for location, path in locations.items():
    wide = pd.read_csv(path, sep=";", dtype={"Leto": int})

    long = wide.melt(id_vars="Leto", var_name="column", value_name="yield_kg_ha")

    # Column format: "{management}-{fertilization} {product}"
    # e.g. "A-N0 Zrnje" -> management=A, fertilization=N0, product=Zrnje
    split = long["column"].str.extract(r"^(?P<management>[A-C])-(?P<fertilization>N\d) (?P<product>.+)$")
    long = pd.concat([long.drop(columns="column"), split], axis=1)

    long["location"] = location
    long = long.rename(columns={"Leto": "year"})
    long = long[["location", "year", "management", "fertilization", "product", "yield_kg_ha"]]
    long = long.dropna(subset=["yield_kg_ha"]).reset_index(drop=True)

    out_path = config.INTERIM_YIELD_DIR / f"maize-{location}.csv"
    long.to_csv(out_path, index=False)
    print(f"Saved {len(long)} rows to {out_path}")

Saved 620 rows to C:\Users\rokuk\Documents\Code\aquacrop-slovenia\data\interim\yield\maize-jablje.csv
Saved 560 rows to C:\Users\rokuk\Documents\Code\aquacrop-slovenia\data\interim\yield\maize-rakican.csv


In [6]:
long.head(20)

,location,year,management,fertilization,product,yield_kg_ha
0,rakican,1993,A,N0,Zrnje,4954.0
1,rakican,1994,A,N0,Zrnje,6283.0
2,rakican,1995,A,N0,Zrnje,6685.0
3,rakican,1996,A,N0,Zrnje,2884.0
4,rakican,1997,A,N0,Zrnje,6928.0
5,rakican,1999,A,N0,Zrnje,4075.0
6,rakican,2000,A,N0,Zrnje,5536.0
7,rakican,2001,A,N0,Zrnje,3922.0
8,rakican,2002,A,N0,Zrnje,9077.0
9,rakican,2003,A,N0,Zrnje,3241.0
